# Model Architecture Visualization

Ten notebook służy do wizualizacji i analizy struktury modeli używanych w projekcie StatsBomb Scout.

Dostępne modele:
- **LSTM**: Podstawowy model LSTM z dwiema warstwami
- **Attention LSTM**: LSTM z mechanizmem uwagi (attention)
- **BiGRU**: Bidirectional GRU z temporal attention pooling
- **Transformer**: Model oparty na architekturze Transformer

In [1]:
import sys
sys.path.append('src')

from src.ml.models.lstm import LSTMSequenceModel
from src.ml.models.attention_lstm import AttentionLSTMModel
from src.ml.models.bigru import build_seq_value_model
from src.ml.models.transformer import TransformerSequenceModel

from tensorflow import keras
import tensorflow as tf

# Parametry wejściowe (dostosuj według potrzeb)
INPUT_SHAPE = (30, 20)  # (max_sequence_length, num_features)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


## 1. LSTM Model

Podstawowy model LSTM składający się z:
- Warstwy maskującej (Masking) - ignoruje padding
- Dwóch warstw LSTM (128 jednostek, 64 jednostki)
- Warstw Dropout dla regularyzacji
- Warstwy Dense (64 jednostki) z aktywacją ReLU
- Wyjściowej warstwy Dense (1 jednostka) - regresja

In [2]:
# LSTM Model
lstm_model = LSTMSequenceModel(
    input_shape=INPUT_SHAPE,
    lstm_units=128,
    dropout=0.2
)
lstm_model.build()

print("=" * 80)
print("LSTM MODEL SUMMARY")
print("=" * 80)
lstm_model.model.summary()

LSTM MODEL SUMMARY


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 30, 20)    │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 30, 20)    │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 30)        │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 30, 128)   │     76,288 │ masking[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 128)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ dropout[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         65 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 125,761 (491.25 KB)

 Trainable params: 125,761 (491.25 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Wizualizacja graficzna LSTM
keras.utils.plot_model(
    lstm_model.model,
    to_file='models/lstm_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/lstm_architecture.png")

Diagram zapisany jako: models/lstm_architecture.png


## 2. Attention LSTM Model

LSTM z mechanizmem uwagi:
- Warstwy maskującej (Masking)
- Bidirectional LSTM (2 × 128 = 256 jednostek)
- Drugiej warstwy LSTM (128 jednostek)
- **Custom AttentionLayer** - oblicza wagi uwagi dla każdego kroku czasowego
- Warstwy Dense (64 jednostki)
- Dwa wyjścia: wartość predykcji oraz wagi uwagi

In [4]:
# Attention LSTM Model
attention_lstm_model = AttentionLSTMModel(
    input_shape=INPUT_SHAPE,
    lstm_units=128,
    dropout=0.2
)
attention_lstm_model.build()

print("=" * 80)
print("ATTENTION LSTM MODEL SUMMARY")
print("=" * 80)
attention_lstm_model.model.summary()

ATTENTION LSTM MODEL SUMMARY


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 30, 20)    │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_1 (Masking) │ (None, 30, 20)    │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_1 (Any)         │ (None, 30)        │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 30, 256)   │    152,576 │ masking_1[0][0],  │
│ (Bidirectional)     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 30, 256)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 30, 128)   │    197,120 │ dropout_2[0][0],  │
│                     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 30, 128)   │          0 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ [(None, 128),     │        158 │ dropout_3[0][0],  │
│ (AttentionLayer)    │ (None, 30)]       │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ attention_weight… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         65 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 358,175 (1.37 MB)

 Trainable params: 358,175 (1.37 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Wizualizacja graficzna Attention LSTM
keras.utils.plot_model(
    attention_lstm_model.model,
    to_file='models/attention_lstm_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/attention_lstm_architecture.png")

Diagram zapisany jako: models/attention_lstm_architecture.png


## 3. BiGRU Model

Bidirectional GRU z temporal attention:
- Warstwy maskującej (Masking)
- Bidirectional GRU (2 × 128 = 256 jednostek) z regularyzacją L2 i constraints
- Layer Normalization
- **TemporalAttentionPooling** - aggreguje sekwencję z wagami uwagi
- Dense (128 jednostek) z Batch Normalization
- Wyjście: wartość predykcji i wagi uwagi (opcjonalnie)

In [6]:
# BiGRU Model
bigru_model = build_seq_value_model(
    input_shape=INPUT_SHAPE,
    rnn_units=128,
    attn_hidden=64,
    dropout=0.2,
    recurrent_dropout=0.15,
    l2_reg=0.01,
    return_attention=True  # Zwróć również wagi uwagi
)

print("=" * 80)
print("BiGRU MODEL SUMMARY")
print("=" * 80)
bigru_model.summary()

BiGRU MODEL SUMMARY


Model: "bigru_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 30, 20)    │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_2 (Masking) │ (None, 30, 20)    │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_2 (Any)         │ (None, 30)        │          0 │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bigru               │ (None, 30, 256)   │    115,200 │ masking_2[0][0],  │
│ (Bidirectional)     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 30, 256)   │        512 │ bigru[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_pool           │ [(None, 256),     │     16,513 │ layer_normalizat… │
│ (TemporalAttention… │ (None, 30)]       │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     32,896 │ attn_pool[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ (None, 30)        │          0 │ attn_pool[0][1]   │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │        129 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 165,762 (647.51 KB)

 Trainable params: 165,506 (646.51 KB)

 Non-trainable params: 256 (1.00 KB)

In [7]:
# Wizualizacja graficzna BiGRU
keras.utils.plot_model(
    bigru_model,
    to_file='models/bigru_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/bigru_architecture.png")

Diagram zapisany jako: models/bigru_architecture.png


## 4. Transformer Model

Model oparty na architekturze Transformer:
- Warstwy maskującej (Masking)
- Projekcja do d_model wymiarów
- **Positional Encoding** - dodaje informację o pozycji w sekwencji
- **Transformer Encoder Blocks** (domyślnie 2):
  - Multi-Head Self-Attention (4 głowice)
  - Feed-Forward Network (512 jednostek)
  - Layer Normalization i residual connections
- Global Average Pooling - agregacja sekwencji
- Dense (64 jednostki)
- Dwa wyjścia: wartość predykcji oraz wagi uwagi

In [8]:
# Transformer Model
transformer_model = TransformerSequenceModel(
    input_shape=INPUT_SHAPE,
    num_heads=4,
    d_model=128,
    ff_dim=512,
    num_blocks=2,
    dropout=0.1,
    true_attention=False
)
transformer_model.build()

print("=" * 80)
print("TRANSFORMER MODEL SUMMARY")
print("=" * 80)
transformer_model.model.summary()

TRANSFORMER MODEL SUMMARY


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_3 (Masking) │ (None, 30, 20)    │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 30, 20)    │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 30, 128)   │      2,688 │ masking_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_3 (Any)         │ (None, 30)        │          0 │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_positional_enc… │ (None, 30, 128)   │      3,840 │ dense_5[0][0],    │
│ (AddPositionalEnco… │                   │            │ any_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 128)   │     66,048 │ add_positional_e… │
│ (MultiHeadAttentio… │                   │            │ add_positional_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 30, 128)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 30, 128)   │          0 │ add_positional_e… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 30, 512)   │     66,048 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 30, 128)   │     65,664 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 30, 128)   │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 30, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logical_or          │ (None, 30)        │          0 │ any_3[0][0],      │
│ (LogicalOr)         │                   │            │ any_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 128)   │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logical_or_1        │ (None, 30)        │          0 │ logical_or[0][0], │
│ (LogicalOr)         │                   │            │ logical_or[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights_… │ [(None, 30, 128), │     66,206 │ layer_normalizat… │
│ (AttentionWeightsL… │ (None, 30)]       │            │ logical_or_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 30, 128)   │          0 │ attention_weight… │
│ (Dropout)           │                   │            │                 

 Total params: 411,551 (1.57 MB)

 Trainable params: 411,551 (1.57 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Wizualizacja graficzna Transformer
keras.utils.plot_model(
    transformer_model.model,
    to_file='models/transformer_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/transformer_architecture.png")

Diagram zapisany jako: models/transformer_architecture.png


## 5. Porównanie liczby parametrów

Zestawienie wszystkich modeli z liczbą parametrów trenowalnych.

In [10]:
import pandas as pd

# Zbierz statystyki wszystkich modeli
models_comparison = [
    {
        'Model': 'LSTM',
        'Total Parameters': lstm_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in lstm_model.model.trainable_weights]),
        'Architecture': 'LSTM → LSTM → Dense'
    },
    {
        'Model': 'Attention LSTM',
        'Total Parameters': attention_lstm_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in attention_lstm_model.model.trainable_weights]),
        'Architecture': 'Bi-LSTM → LSTM → Attention → Dense'
    },
    {
        'Model': 'BiGRU',
        'Total Parameters': bigru_model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in bigru_model.trainable_weights]),
        'Architecture': 'Bi-GRU → Temporal Attention → Dense'
    },
    {
        'Model': 'Transformer',
        'Total Parameters': transformer_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in transformer_model.model.trainable_weights]),
        'Architecture': 'Positional Encoding → Transformer Blocks → Pooling → Dense'
    }
]

df_comparison = pd.DataFrame(models_comparison)
df_comparison['Total Parameters'] = df_comparison['Total Parameters'].apply(lambda x: f"{x:,}")
df_comparison['Trainable Parameters'] = df_comparison['Trainable Parameters'].apply(lambda x: f"{x:,}")

print("\n" + "=" * 100)
print("MODEL COMPARISON")
print("=" * 100)
print(df_comparison.to_string(index=False))
print("=" * 100)


MODEL COMPARISON
         Model Total Parameters Trainable Parameters                                               Architecture
          LSTM          125,761              125,761                                        LSTM → LSTM → Dense
Attention LSTM          358,175              358,175                         Bi-LSTM → LSTM → Attention → Dense
         BiGRU          165,762              165,506                        Bi-GRU → Temporal Attention → Dense
   Transformer          411,551              411,551 Positional Encoding → Transformer Blocks → Pooling → Dense


## 6. Analiza poszczególnych warstw

Szczegółowa analiza konkretnych warstw w wybranym modelu.

In [11]:
# Przykład: Analiza warstw Attention LSTM
print("\nDetailed Layer Analysis - Attention LSTM Model:")
print("=" * 80)

for i, layer in enumerate(attention_lstm_model.model.layers):
    print(f"\nLayer {i}: {layer.name}")
    print(f"  Type: {type(layer).__name__}")
    print(f"  Output Shape: {layer.output_shape}")
    if hasattr(layer, 'units'):
        print(f"  Units: {layer.units}")
    if hasattr(layer, 'activation'):
        print(f"  Activation: {layer.activation}")
    
    # Liczba parametrów w warstwie
    trainable_params = sum([tf.size(w).numpy() for w in layer.trainable_weights])
    non_trainable_params = sum([tf.size(w).numpy() for w in layer.non_trainable_weights])
    
    if trainable_params > 0 or non_trainable_params > 0:
        print(f"  Trainable params: {trainable_params:,}")
        print(f"  Non-trainable params: {non_trainable_params:,}")


Detailed Layer Analysis - Attention LSTM Model:

Layer 0: sequence_input
  Type: InputLayer


AttributeError: 'InputLayer' object has no attribute 'output_shape'

## 7. Testowe przewidywanie

Sprawdzenie czy model działa poprawnie na przykładowych danych.

In [ ]:
import numpy as np

# Wygeneruj przykładowe dane
batch_size = 2
sample_input = np.random.randn(batch_size, INPUT_SHAPE[0], INPUT_SHAPE[1]).astype(np.float32)

# Dodaj trochę padding'u (zera na końcu)
sample_input[:, -5:, :] = 0.0

print("\nTest Prediction:")
print("=" * 80)
print(f"Input shape: {sample_input.shape}")

# LSTM (pojedyncze wyjście)
lstm_pred = lstm_model.model.predict(sample_input, verbose=0)
print(f"\nLSTM prediction shape: {lstm_pred.shape}")
print(f"LSTM prediction values: {lstm_pred.flatten()}")

# Attention LSTM (dwa wyjścia: value + attention_weights)
attention_lstm_pred = attention_lstm_model.model.predict(sample_input, verbose=0)
print(f"\nAttention LSTM prediction:")
print(f"  Value shape: {attention_lstm_pred['value'].shape}")
print(f"  Value: {attention_lstm_pred['value'].flatten()}")
print(f"  Attention weights shape: {attention_lstm_pred['attention_weights'].shape}")
print(f"  Attention weights sum (should be ~1.0): {attention_lstm_pred['attention_weights'].sum(axis=1)}")

## 8. Eksport do LaTeX/dokumentacji

Przygotowanie tabel do publikacji naukowej.

In [ ]:
# Eksport do LaTeX
latex_table = df_comparison.to_latex(index=False, caption="Porównanie architektur modeli", label="tab:model_comparison")
print("\nLaTeX Table:")
print(latex_table)

# Zapisz do pliku
with open('models/model_comparison_table.tex', 'w') as f:
    f.write(latex_table)
print("\nTabela LaTeX zapisana jako: models/model_comparison_table.tex")